# Using the onanana package

Examples of local vs cloud model routing using `src/onanana/` modules directly.

In [1]:
import sys
sys.path.insert(0, "..")

import httpx
from src.onanana.keys_manager import KeysManager
from src.onanana.providers.ollama import OllamaProvider

In [2]:
km = KeysManager("../secrets/keys.txt", cloud_base_url="https://api.cloud.com")
km.load_keys()
print(f"Loaded {len(km._keys)} key(s)")

Loaded 2 key(s)


In [3]:
client = httpx.AsyncClient(timeout=300.0)
provider = OllamaProvider(
    local_base_url="http://192.168.100.66:11434",
    cloud_base_url="https://api.ollama.com",
    keys_manager=km,
    client=client,
)

## Chat with local model

In [4]:
body = {"model": "gemma4:e4b", "messages": [{"role": "user", "content": "hi"}], "stream": False}
resp = await provider.proxy_request("/api/chat", body)
await resp.aread()
print("Status:", resp.status_code)
print("Body:", resp.text)

Status: 200
Body: {"model":"gemma4:e4b","created_at":"2026-07-14T04:50:11.2832423Z","message":{"role":"assistant","content":"Hi! How can I help you today? 😊"},"done":true,"done_reason":"stop","total_duration":44034953300,"load_duration":41915094900,"prompt_eval_count":17,"prompt_eval_duration":664666000,"eval_count":11,"eval_duration":1450353000}


## Chat with cloud model (via -cloud suffix)

In [5]:
body = {"model": "gemma4:31b-cloud", "messages": [{"role": "user", "content": "hi"}], "stream": False}
resp = await provider.proxy_request("/api/chat", body)
await resp.aread()
print("Status:", resp.status_code)
print("Body:", resp.text)

Status: 200
Body: {"model":"gemma4:31b","created_at":"2026-07-14T04:50:09.759755925Z","message":{"role":"assistant","content":"Hello! How can I help you today?"},"done":true,"done_reason":"stop","total_duration":321579275,"prompt_eval_count":14,"eval_count":10}



## Chat with cloud model (via ?source= param)

In [6]:
body = {"model": "gemma4:e4b", "messages": [{"role": "user", "content": "hi"}], "stream": False}
resp = await provider.proxy_request("/api/chat", body, source="cloud")
await resp.aread()
print("Status:", resp.status_code)
print("Body:", resp.text)

Status: 404
Body: {"error": "model 'gemma4:e4b' not found"}



In [7]:
body = {"model": "gemma4:31b-cloud", "messages": [{"role": "user", "content": "hi"}], "stream": False}
resp = await provider.proxy_request("/api/chat", body, source="cloud")
await resp.aread()
print("Status:", resp.status_code)
print("Body:", resp.text)

Status: 200
Body: {"model":"gemma4:31b","created_at":"2026-07-14T04:50:31.257205224Z","message":{"role":"assistant","content":"Hello! How can I help you today?"},"done":true,"done_reason":"stop","total_duration":20756573462,"prompt_eval_count":14,"eval_count":10}



## Generate local

In [ ]:
body = {"model": "gemma4:e4b", "prompt": "hello"}
resp = await provider.proxy_request("/api/generate", body)
await resp.aread()
print("Status:", resp.status_code)
print("Body:", resp.text)

## Generate cloud

In [ ]:
body = {"model": "gemma4:31b-cloud", "prompt": "hello"}
resp = await provider.proxy_request("/api/generate", body)
await resp.aread()
print("Status:", resp.status_code)
print("Body:", resp.text)

## Tags (local)

In [13]:
resp = await provider.proxy_get("api/tags", source="local")
print(resp.status_code, resp.json())

200 {'models': [{'name': 'gemma4:12b', 'model': 'gemma4:12b', 'modified_at': '2026-06-08T09:54:44.7670799+07:00', 'size': 7556508396, 'digest': '4eb23ef187e2c5462566d6a1d3bbbc2f1346d0b4327cbb66d58fffbcc9b2b05c', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma4', 'families': ['gemma4'], 'parameter_size': '11.9B', 'quantization_level': 'Q4_K_M', 'context_length': 262144, 'embedding_length': 3840}, 'capabilities': ['completion', 'tools', 'thinking', 'vision']}, {'name': 'gemma4:e4b', 'model': 'gemma4:e4b', 'modified_at': '2026-06-08T09:29:56.8371448+07:00', 'size': 9608350718, 'digest': 'c6eb396dbd5992bbe3f5cdb947e8bbc0ee413d7c17e2beaae69f5d569cf982eb', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma4', 'families': ['gemma4'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M'}, 'capabilities': ['completion', 'tools', 'thinking']}, {'name': 'hf.co/jinaai/jina-embeddings-v5-omni-small-retrieval-GGUF:Q4_K_M', 'model': 'hf.co/jinaai/jina-embeddings

## Tags (cloud)

In [14]:
resp = await provider.proxy_get("api/tags", source="cloud")
print(resp.status_code, resp.json())

200 {'models': [{'name': 'glm-5.1', 'model': 'glm-5.1', 'modified_at': '2026-04-07T08:00:00-07:00', 'size': 1507728316928, 'digest': '882e35812821', 'details': {'parent_model': '', 'format': '', 'family': '', 'families': None, 'parameter_size': '', 'quantization_level': ''}}, {'name': 'glm-5.2', 'model': 'glm-5.2', 'modified_at': '2026-06-16T08:00:00-07:00', 'size': 0, 'digest': 'f553240f6dc4', 'details': {'parent_model': '', 'format': '', 'family': '', 'families': None, 'parameter_size': '', 'quantization_level': ''}}, {'name': 'kimi-k2.5', 'model': 'kimi-k2.5', 'modified_at': '2026-01-26T00:00:00Z', 'size': 1118481408000, 'digest': '89c148d8ace8', 'details': {'parent_model': '', 'format': '', 'family': '', 'families': None, 'parameter_size': '', 'quantization_level': ''}}, {'name': 'deepseek-v3.2', 'model': 'deepseek-v3.2', 'modified_at': '2025-12-02T00:00:00Z', 'size': 688586727753, 'digest': 'a403a93c7b13', 'details': {'parent_model': '', 'format': '', 'family': '', 'families': Non

## Version (local)

In [15]:
resp = await provider.proxy_get("api/version", source="local")
print(resp.status_code, resp.json())

200 {'version': '0.31.2'}


## Version (cloud)

In [16]:
resp = await provider.proxy_get("api/version", source="cloud")
print(resp.status_code, resp.json())

200 {'version': '0.0.0'}


## OpenAI-compatible response format

Using `to_openai_completion()` from `src/onanana/openai.py` to convert Ollama responses to OpenAI `/v1/chat/completions` format. This is the same conversion used by the `/v1/chat/completions` API endpoint.

In [ ]:
from src.onanana.openai import to_openai_completion

body = {"model": "gemma4:e4b", "messages": [{"role": "user", "content": "hello"}], "stream": False}
resp = await provider.proxy_request("/api/chat", body)
await resp.aread()
ollama_data = resp.json()

openai_resp = to_openai_completion(ollama_data, "gemma4:e4b")
print("OpenAI completion:")
print(openai_resp["choices"][0]["message"]["content"])

In [ ]:
# Cloud model via -cloud suffix
body = {"model": "gemma4:31b-cloud", "messages": [{"role": "user", "content": "hello"}], "stream": False}
resp = await provider.proxy_request("/api/chat", body)
await resp.aread()
ollama_data = resp.json()

openai_resp = to_openai_completion(ollama_data, "gemma4:31b-cloud")
print("OpenAI completion (cloud):")
print(openai_resp["choices"][0]["message"]["content"])

In [8]:
await client.aclose()
await km.close()
print("done")   

done
